In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean
from scipy.stats import pearsonr

# 1. Load dataset
df = pd.read_csv("Lab2_titanic.csv")
print("Original Data:\n", df.head())


# 2. Imputation
num_imputer = SimpleImputer(strategy="mean")

for col in df.columns:
    df[col] = num_imputer.fit_transform(df[[col]])

# ---- Missing values after imputation ----
plt.figure(figsize=(6,4))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Missing Values After Imputation")
plt.show()

print("\nAfter Imputation:\n", df.head())

# 3. Encoding categorical data (Label + OneHot)
label_enc = LabelEncoder()
for col in df.select_dtypes(include=["object"]).columns:
    df[col + "_label"] = label_enc.fit_transform(df[col])

df_onehot = pd.get_dummies(df.select_dtypes(include=["object"]))
df = pd.concat([df, df_onehot], axis=1)

print("\nAfter Encoding:\n", df.head())

# 4. Scaling (Min-Max & Z-score)
scaler_minmax = MinMaxScaler()
scaler_zscore = StandardScaler()

numeric_cols = df.select_dtypes(include=[np.number]).columns
df_minmax = pd.DataFrame(scaler_minmax.fit_transform(df[numeric_cols]), columns=numeric_cols)
df_zscore = pd.DataFrame(scaler_zscore.fit_transform(df[numeric_cols]), columns=numeric_cols)

# ---- Distribution before vs after normalization ----
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
sns.histplot(df[numeric_cols[0]], kde=True)
plt.title("Original Distribution: " + numeric_cols[0])

plt.subplot(1,2,2)
sns.histplot(df_minmax[numeric_cols[0]], kde=True, color="orange")
plt.title("Min-Max Normalized: " + numeric_cols[0])
plt.show()

# ---- Correlation heatmap ----
plt.figure(figsize=(8,6))
sns.heatmap(df_minmax.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Min-Max Normalized Data)")
plt.show()

# 5. Similarity & Dissimilarity Measures (between first 2 rows)
x = df_minmax.iloc[0]
y = df_minmax.iloc[1]

# Pearson
pearson_corr, _ = pearsonr(x, y)
print("\nPearson Correlation:", pearson_corr)

# Cosine
cos_sim = cosine_similarity([x], [y])[0][0]
print("Cosine Similarity:", cos_sim)

# Jaccard (on one-hot features)
x_bin = df_onehot.iloc[0].to_numpy()
y_bin = df_onehot.iloc[1].to_numpy()
jaccard_sim = np.sum((x_bin & y_bin)) / np.sum((x_bin | y_bin))
print("Jaccard Similarity:", jaccard_sim)

# Euclidean
eu_dist = euclidean(x, y)
print("Euclidean Distance:", eu_dist)

Original Data:
    PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   N

ValueError: Cannot use mean strategy with non-numeric data:
could not convert string to float: 'Braund, Mr. Owen Harris'